In [ ]:
# main.py
import numpy as np
from DispersionModels import gaussian_plume_gp
from Sensors import PIDGasSensor
from MovementStrategies import ParameterizedSweepStrategy
from SimulationClasses import World, UAV, DispersionModelConfig
from Visualization import plot_mission_results
from ParticleFilter import ParticleFilter, ParticleFilterConfig

def main():
    # --- 1. Global Parameters & World Setup ---
    true_pitch = 0.0
    true_yaw = np.arctan2(1.0, 1.0) # Retorna o ângulo correto no plano XY
    wind_angles_gt = (true_pitch, true_yaw)
    source_position=(10.0, 10.0, 1.5)

    # Nova estrutura de configuração do ambiente (Ground Truth)
    env_config = DispersionModelConfig(
        source_position=source_position,
        q=1.0,
        u=5.0,
        zetas=(0.3, 0.3),
        wind_angles=wind_angles_gt
    )

    # O World recebe a função pura e o struct de configuração
    world = World(
        dispersion_model=gaussian_plume_gp, 
        model_config=env_config
    )

    # --- 2. UAV & Sensors Setup ---
    strategy = ParameterizedSweepStrategy(x_max=40.0, y_max=40.0, num_sectors=12)
    uav1 = UAV("UAV_01", world, strategy)

    # Adding multiple named sensors to the UAV
    uav1.add_sensor(PIDGasSensor(name="MQ4", min_resolution=1e-4, noise_std=0.001))
    uav1.add_sensor(PIDGasSensor(name="CO2_Sensor", min_resolution=1e-3, noise_std=0.005))

    # Initialize UAV position
    t_progression = 0.0
    t_step = 0.005
    world.register_uav(uav1.id, strategy.compute_next_step(t_progression))

    # --- 3. Estimator Setup ---
    pf_estimator = ParticleFilter()
    pf_config = ParticleFilterConfig(
        num_particles=3000,
        n_eff_threshold=0.5,
        resample_function="systematic",
        target_sensor_name="MQ4"
    )

    # Definindo as distribuições a priori de forma hierarquizada
    # O Parametro dist não faz nada por hora...
    world_prior_params = {
        "x_s": {"dist": "uniform", "low": 0.0, "high": 40.0},
        "y_s": {"dist": "uniform", "low": 0.0, "high": 40.0},
        "q":   {"dist": "gamma", "shape": 1.5, "scale": 1.0},
        "u_s": {"dist": "normal", "mean": 1.0, "std": 0.2}, 
        "pitch": {"low": -np.pi/2, "high": np.pi/2}, # Pitch geralmente é restrito a [-90, 90] graus
        "yaw":   {"low": -np.pi, "high": np.pi},
        "zeta1": {"dist": "uniform", "low": 1.0, "high": 25.0},
        "zeta2": {"dist": "uniform", "low": 1.0, "high": 25.0}
    }
    
    # The memory is created here, not inside the class!
    # Inicializamos usando o dicionário e a config:
    pf_memory_particles = pf_estimator.initialize_particles(pf_config, world_prior_params)

    # --- Tracking Data for Plotting ---
    measurement_history = []
    trajectory_history = []

    print("Starting Mission...")

    # --- 4. Mission Loop ---
    mission_active = True
    while mission_active:

        # STEP 1: Read measurements from all UAVs
        uav_data_snapshot = {}

        pos = uav1.get_position()
        real_conc = world.get_real_concentration(pos)

        sensor_data = {
            "MQ4": uav1.get_sensor("MQ4").read_measurement(real_conc),
            "CO2_Sensor": uav1.get_sensor("CO2_Sensor").read_measurement(real_conc)
        }

        uav_data_snapshot[uav1.id] = {
            "position": pos,
            "sensors": sensor_data
        }

        # Save history for visualization (Using MQ4 data for the plot)
        measurement_history.append(sensor_data["MQ4"])
        trajectory_history.append(pos)

        # STEP 2: Execute Estimator
        pf_memory_particles = pf_estimator.step(
            uav_data=uav_data_snapshot,
            memory=pf_memory_particles,
            config=pf_config
        )

        # STEP 3: Stop Criteria
        if t_progression >= 1.0:
            print("Mission Objective Reached: Path Complete.")
            mission_active = False
            break

        # STEP 4: Movement
        t_progression = min(t_progression + t_step, 1.0)
        next_pos = uav1.strategy.compute_next_step(t_progression)
        uav1.set_position(next_pos)

    # --- 5. Final Visualization ---
    grid_x, grid_y = np.meshgrid(np.linspace(0, 50, 200), np.linspace(0, 50, 200))
    Z = gaussian_plume_gp((grid_x, grid_y, 2.5), source_position, 1.0, 5.0, (0.3, 0.3), wind_angles_gt)

    plot_mission_results(
        grid_x, grid_y, Z,
        trajectory_history, measurement_history,
        source_position, wind_angles_gt
    )

if __name__ == "__main__":
    main()